In [0]:
source_dir="/Volumes/eccom_catalog/default/ecomm_data/orders_data/Source/"
archive_dir="/Volumes/eccom_catalog/default/ecomm_data/orders_data/Archive/"
order_stage_table="eccom_catalog.default.orders_stage"
error_table="eccom_catalog.default.error_table"

In [0]:
#Read orders csv file
from pyspark.sql.functions import current_timestamp,lit,col
from datetime import datetime
try:
    df_order=spark.read.csv(source_dir, header=True, inferSchema=True, dateFormat="yyyy-MM-dd", timestampFormat="yyyy-MM-dd HH:mm:ss")
    df_order=df_order.withColumn("processed_timestamp",current_timestamp()).withColumn("batch_id", lit(datetime.now().strftime("%Y%m%d_%H%M%S"))).withColumn("source_system", lit("ecommerce_orders"))
    print("Successfully read orders file")

except Exception as e:
    print(f"Failed to read orders file: {str(e)}")
    raise

In [0]:
#Apply Data Quality checks
try:
    total_records=df_order.count()
    total_null_order_ids=df_order.filter(col('order_id').isNull()).count()
    total_invalid_amount=df_order.filter(col('order_amount')<=0).count()
    print("Data Quality checks")
    print(f"Total records: {total_records}")
    print(f"Total null order ids: {total_null_order_ids}")
    print(f"Total invalid amount: {total_invalid_amount}")
    
    valid_records=df_order.filter((col('order_id').isNotNull()) & (col('order_amount')>0))
    invalid_records=df_order.filter((col('order_id').isNull()) | (col('order_amount')<=0))

    total_valid_records=valid_records.count()
    total_invalid_records=invalid_records.count()

    print(f"Total valid records: {total_valid_records}")
    print(f"Total invalid records: {total_invalid_records}")

except Exception as e:
    print(f"Failed Data Quality checks: {str(e)}")
    raise

In [0]:
#Writing valid records in stage table
try:
    valid_records.write.mode('overwrite').format('delta').saveAsTable(order_stage_table)
    print(f"Successfully write {total_valid_records} records in staging table")
    if total_invalid_records>0:
        invalid_records.withColumn('error_reason',lit('Data Quality validation fail')).withColumn('error_timestamp', current_timestamp())\
            .write.mode('append').format('delta').saveAsTable(error_table)
        print(f"logged {total_invalid_records} invalid records to error table")
    else:
        print("No invalid records found")    

except Exception as e:
    print(f"Error writing to staging table: {str(e)}")
    raise

In [0]:
#Archive processed files
try:
    files=dbutils.fs.ls(source_dir)
    archive_count=0
    for file in files:
        if file.name.endswith('.csv'):
            source=file.path
            archive_path=archive_dir+file.name
            dbutils.fs.mv(source,archive_path)
            archive_count+=1
    print(f"Archived {archive_count} files")

except Exception as e:
    print(f"Failed to archive processed files: {str(e)}")
    raise

In [0]:
#Log processing summary
import json
try:
    processing_summary={
        'task': 'orders_stage_load',
        'timestamp': datetime.now().isoformat(),
        'total_records': total_records,
        'valid_records': total_valid_records,
        'invalid_records': total_invalid_records,
        'archived_files': archive_count,
        'status': 'SUCCESS' if total_invalid_records==0 else 'SUCCESS_WITH_WARNINGS'
        }
    print(f"Processing Summary: {json.dumps(processing_summary)}") 
    df_processing=spark.createDataFrame([processing_summary])
    df_processing.write.mode('append').format('delta').saveAsTable('eccom_catalog.default.processing_log')  
    print("Successfully logged processing summary")
    
except Exception as e:
    print(f"Failed to log processing summary: {str(e)}")   